# Half Your Network Is Already Dead

**Topic:** Initialization and ReLU &nbsp;·&nbsp; **Difficulty:** Medium

A ReLU unit whose pre-activation is negative for every input in the dataset
outputs zero forever, and its gradient is zero too, so it can never recover.

Initialize a wide layer three ways and count how many units are dead before a
single training step. The bias initialization is where this usually goes
wrong.

## The data (created in the next cell)

- `X` `(2000, 100)`: standardized inputs.
- A single layer of 512 ReLU units.
- Three initializations, all seeded with `np.random.default_rng(9)`.

## Your task

With **NumPy only**, for each of the three schemes below, build `W` `(100, 512)`
and `b` `(512,)`, compute `A = X @ W + b`, and count the **dead** units, those with
`A[:, j] <= 0` for every row:

1. `"he"`: `W ~ N(0, 2/100)`, `b = 0`.
2. `"negative_bias"`: the same `W`, but `b = -5.0` everywhere.
3. `"too_small"`: `W` with standard deviation `1e-4`, `b = -0.01`.

Also record, for each scheme, the fraction of ReLU outputs that are zero, which is the fraction of the 2000 by 512 pre-activations at or below 0.

Set `answers` to a dict with `"dead_he"`, `"dead_negbias"`, `"dead_small"` (ints)
and `"zero_frac_he"`, `"zero_frac_negbias"`, `"zero_frac_small"` (floats).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks He initialization leaves no dead units, the negative-bias scheme kills
most of them, and the tiny-weight scheme kills all 512.

In [ ]:
import numpy as np

rng = np.random.default_rng(2121)
X = rng.normal(0, 1, (2000, 100))
d_in, d_out = 100, 512
print(f"X {X.shape}, layer {d_in} -> {d_out} ReLU units")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("dead_he", [], 0.0, 0.0, 1e-06),
    ("zero_frac_he", [], 0.499794921875, -0.0836846695331177, 1e-06),
    ("dead_negbias", [], 341.0, -57.09636305174421, 1e-06),
    ("zero_frac_negbias", [], 0.9997353515625, -0.1673937026055004, 1e-06),
    ("dead_small", [], 512.0, -85.72826358502357, 1e-06),
    ("zero_frac_small", [], 1.0, -0.16743801481449916, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "dead_he" in answers:
    _g = (int(answers["dead_he"]) == 0 and int(answers["dead_small"]) == 512
          and int(answers["dead_negbias"]) > 256)
    print(f"  {'He alive, tiny-W dead':<24} {'ok' if _g else 'wrong'}  "
          f"(he {int(answers['dead_he'])}, negbias {int(answers['dead_negbias'])}, "
          f"small {int(answers['dead_small'])} of 512)")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

With He initialization the pre-activations are centred at zero with unit-ish
scale, so roughly half of each unit's inputs land positive and no unit is dead.
Subtracting 5.0 from every bias shifts the whole distribution left by more than
three standard deviations of the pre-activation, so a unit dies whenever its
largest pre-activation over 2000 inputs is still below that, which happens to
most of them. The tiny-weight scheme is worse in a different way: the
pre-activations are all within about 0.005 of the bias, so a single small
negative bias kills the entire layer at once.

```python
import numpy as np

rng = np.random.default_rng(9)
W_he = rng.normal(0, np.sqrt(2.0 / d_in), (d_in, d_out))
W_small = rng.normal(0, 1e-4, (d_in, d_out))

def stats(W, b):
    A = X @ W + b
    dead = int((A <= 0).all(axis=0).sum())
    zero_frac = float((np.maximum(A, 0.0) == 0).mean())
    return dead, zero_frac

d1, z1 = stats(W_he, np.zeros(d_out))
d2, z2 = stats(W_he, np.full(d_out, -5.0))
d3, z3 = stats(W_small, np.full(d_out, -0.01))

answers = {
    "dead_he": d1, "zero_frac_he": z1,
    "dead_negbias": d2, "zero_frac_negbias": z2,
    "dead_small": d3, "zero_frac_small": z3,
}
```